# P1-A7: Multi-Model AI Systems

## Part 1: Model Benchmarking

**Objective:** Compare 15 language models across five parameter-size categories using 25 prompts across five task types.

We will measure response latency, generation speed, and output quality to evaluate the trade-offs between model size, performance, and reliability.

## Part 2: Scaling Simulation

Estimate token demand, infrastructure requirements, and costs at different daily active user levels.

## Part 3: Intelligent Model Routing

Design a routing system that selects a suitable model based on task complexity, latency requirements, and cost.

In [5]:
import gc
import json
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print("Device:", DEVICE)

MODEL_CATALOG = [
    # Ultra-light: < 1B
    {"name": "SmolLM2-360M-Instruct", "id": "HuggingFaceTB/SmolLM2-360M-Instruct", "category": "<1B", "params_b": 0.36},
    {"name": "Qwen2.5-0.5B-Instruct", "id": "Qwen/Qwen2.5-0.5B-Instruct", "category": "<1B", "params_b": 0.5},
    {"name": "Qwen3-0.6B", "id": "Qwen/Qwen3-0.6B", "category": "<1B", "params_b": 0.6},

    # Small: 1B–<3B
    {"name": "Qwen2.5-1.5B-Instruct", "id": "Qwen/Qwen2.5-1.5B-Instruct", "category": "1B–<3B", "params_b": 1.5},
    {"name": "SmolLM2-1.7B-Instruct", "id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "category": "1B–<3B",
     "params_b": 1.7},
    {"name": "Qwen2-1.5B-Instruct", "id": "Qwen/Qwen2-1.5B-Instruct", "category": "1B–<3B", "params_b": 1.5},

    # Medium: 3B–<7B
    {"name": "Qwen2.5-3B-Instruct", "id": "Qwen/Qwen2.5-3B-Instruct", "category": "3B–<7B", "params_b": 3.0},
    {"name": "Qwen3-4B-Instruct-2507", "id": "Qwen/Qwen3-4B-Instruct-2507", "category": "3B–<7B", "params_b": 4.0},
    {"name": "Phi-3.5-mini-instruct", "id": "microsoft/Phi-3.5-mini-instruct", "category": "3B–<7B", "params_b": 3.8},

    # Large: 7B–<13B
    {"name": "Mistral-7B-Instruct-v0.3", "id": "mistralai/Mistral-7B-Instruct-v0.3", "category": "7B–<13B",
     "params_b": 7.0},
    {"name": "Qwen2.5-7B-Instruct", "id": "Qwen/Qwen2.5-7B-Instruct", "category": "7B–<13B", "params_b": 7.0},
    {"name": "Falcon3-7B-Instruct", "id": "tiiuae/Falcon3-7B-Instruct", "category": "7B–<13B", "params_b": 7.0},

    # Extra-large: 13B+
    {"name": "Qwen2.5-14B-Instruct", "id": "Qwen/Qwen2.5-14B-Instruct", "category": "13B+", "params_b": 14.0},
    {"name": "Qwen3-14B", "id": "Qwen/Qwen3-14B", "category": "13B+", "params_b": 14.0},
    {"name": "DeepSeek-R1-Distill-Qwen-14B", "id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B", "category": "13B+",
     "params_b": 14.0},
]

assert len(MODEL_CATALOG) == 15, "Expected 15 models"
assert len({m["id"] for m in MODEL_CATALOG}) == 15

display(pd.DataFrame(MODEL_CATALOG))
print("Device:", DEVICE)

In [ ]:
print("Using device:", DEVICE)


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
    )

    model = model.to(DEVICE)
    model.eval()
    return tokenizer, model


def generate_text(
        tokenizer,
        model,
        prompt,
        max_new_tokens=180,
        temperature=0.0,
        top_k=50,
        top_p=0.95,
):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    generation_options = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature <= 0:
        generation_options["do_sample"] = False
    else:
        generation_options.update({
            "do_sample": True,
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
        })
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(**inputs, **generation_options)
    elapsed = time.perf_counter() - start
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response, elapsed, len(new_tokens)

In [ ]:
import re


def extract_json(text):
    candidates = [text]
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if match:
        candidates.append(match.group(0))
    for candidate in candidates:
        try:
            value = json.loads(candidate)
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            pass
    return None


In [ ]:
PROMPTS = [
    # 1. Structured extraction
    {
        "id": "SE1",
        "type": "structured_extraction",
        "prompt": 'Extract the name, date, and location from: "Maya has a biology exam on November 12 in Lincoln Hall." Return JSON only.',
        "expected": {"name": "Maya", "date": "November 12", "location": "Lincoln Hall"}
    },
    {
        "id": "SE2",
        "type": "structured_extraction",
        "prompt": 'Extract the task and deadline from: "Submit the database project by Friday at 5 PM." Return JSON only.',
        "expected": {"task": "database project", "deadline": "Friday at 5 PM"}
    },
    {
        "id": "SE3",
        "type": "structured_extraction",
        "prompt": 'Extract the course, credits, and instructor from: "CS 124 is a 3-credit course taught by Professor Lee." Return JSON only.',
        "expected": {"course": "CS 124", "credits": 3, "instructor": "Professor Lee"}
    },
    {
        "id": "SE4",
        "type": "structured_extraction",
        "prompt": 'Extract the product, quantity, and unit price from: "I bought 2 notebooks for $6 each." Return JSON only.',
        "expected": {"product": "notebooks", "quantity": 2, "unit_price": 6}
    },
    {
        "id": "SE5",
        "type": "structured_extraction",
        "prompt": 'Extract all events and their dates from: "The workshop is on March 3, and the final presentation is on March 10." Return JSON only.',
        "expected": {"workshop": "March 3", "final presentation": "March 10"}
    },

    # 2. Classification
    {
        "id": "CL1",
        "type": "classification",
        "prompt": 'Classify the sentiment as positive, negative, or neutral: "The lecture was clear and helpful."',
        "expected": "positive"
    },
    {
        "id": "CL2",
        "type": "classification",
        "prompt": 'Classify this request as academic, administrative, or personal: "I need to change my course registration."',
        "expected": "administrative"
    },
    {
        "id": "CL3",
        "type": "classification",
        "prompt": 'Classify the urgency as low, medium, or high: "My assignment is due in three weeks."',
        "expected": "low"
    },
    {
        "id": "CL4",
        "type": "classification",
        "prompt": 'Classify this statement as a fact, opinion, or prediction: "I think the next exam will be difficult."',
        "expected": "prediction"
    },
    {
        "id": "CL5",
        "type": "classification",
        "prompt": 'Classify this activity as review, practice, or new learning: "I am solving additional problems on a topic I already learned."',
        "expected": "practice"
    },

    # 3. Summarization
    {
        "id": "SU1",
        "type": "summarization",
        "prompt": 'Summarize in exactly two bullet points: "Students must complete weekly readings, submit homework every Thursday, attend discussion sections, and complete a final project during the last two weeks."',
        "expected": "Readings and homework are weekly; discussion sections must be attended; final project occurs in the last two weeks."
    },
    {
        "id": "SU2",
        "type": "summarization",
        "prompt": 'Summarize in one sentence: "Model A was fastest, Model B was most accurate, and Model C used the least memory."',
        "expected": "Model A is fastest, Model B is most accurate, and Model C uses the least memory."
    },
    {
        "id": "SU3",
        "type": "summarization",
        "prompt": 'Summarize the action items: "Finish the interface by Monday, test the API on Tuesday, and fix critical bugs before Friday."',
        "expected": "Finish the interface Monday, test the API Tuesday, and fix critical bugs before Friday."
    },
    {
        "id": "SU4",
        "type": "summarization",
        "prompt": 'Summarize in no more than 40 words: "Quizzes account for 15% of the grade, midterms for 40%, and the cumulative final for 45%."',
        "expected": "Quizzes are 15%, midterms are 40%, and the cumulative final is 45% of the grade."
    },
    {
        "id": "SU5",
        "type": "summarization",
        "prompt": 'Summarize the cause and result: "The server received too many simultaneous requests, causing slow responses and timeouts. The team added caching and rate limiting, reducing timeouts."',
        "expected": "Too many requests caused slow responses and timeouts; caching and rate limiting reduced timeouts."
    },

    # 4. Reasoning and estimation
    {
        "id": "RE1",
        "type": "reasoning_estimation",
        "prompt": "A student studies 2 hours per day for 5 days and then 3 hours on Saturday. How many hours did they study?",
        "expected": 13
    },
    {
        "id": "RE2",
        "type": "reasoning_estimation",
        "prompt": "A project has 4 tasks requiring 2, 3, 5, and 4 hours. If 2 hours of work are completed each day, how many full days are needed?",
        "expected": 7
    },
    {
        "id": "RE3",
        "type": "reasoning_estimation",
        "prompt": "A class has 80 students. If 25% miss a lecture, how many students attended?",
        "expected": 60
    },
    {
        "id": "RE4",
        "type": "reasoning_estimation",
        "prompt": "A model processes 120 tokens in 6 seconds. What is its throughput in tokens per second?",
        "expected": 20
    },
    {
        "id": "RE5",
        "type": "reasoning_estimation",
        "prompt": "A student has 12 hours available and 4 assignments requiring 2, 3, 4, and 5 hours. Can all assignments be completed?",
        "expected": "No. The assignments require 14 hours, which exceeds the available 12 hours."
    },

    # 5. Ambiguity handling
    {
        "id": "AH1",
        "type": "ambiguity_handling",
        "prompt": 'A student says, "Schedule my study session for next Friday." What information should you clarify before scheduling?',
        "expected": "Clarify the intended date, time, duration, and time zone if relevant."
    },
    {
        "id": "AH2",
        "type": "ambiguity_handling",
        "prompt": 'A user asks, "Which model is best?" Explain what you need to clarify.',
        "expected": "Ask about the task, accuracy needs, latency, budget, and deployment constraints."
    },
    {
        "id": "AH3",
        "type": "ambiguity_handling",
        "prompt": 'A student says, "I need more time for my project." Respond helpfully without making assumptions.',
        "expected": "Ask whether they need help planning their work, estimating effort, or requesting an extension."
    },
    {
        "id": "AH4",
        "type": "ambiguity_handling",
        "prompt": 'A user says, "Remind me before the exam." What information is missing?',
        "expected": "Ask for the exam date and time and how long before it the reminder should occur."
    },
    {
        "id": "AH5",
        "type": "ambiguity_handling",
        "prompt": 'A student asks, "Will I pass this course?" What information would help estimate this?',
        "expected": "Ask for current grades, grading weights, remaining assessments, and passing threshold."
    }
]

assert len(PROMPTS) == 25
assert len({p["id"] for p in PROMPTS}) == 25

print("Total prompts:", len(PROMPTS))
print(pd.Series([p["type"] for p in PROMPTS]).value_counts())

In [ ]:
def evaluate_response(tokenizer, model, prompt, response):
    evaluation_prompt = f"""
Evaluate the response to the prompt below.

Prompt:
{prompt}

Response:
{response}

Score each criterion from 1 to 10:
- instruction_adherence: follows the prompt's requirements
- accuracy: factual and logical correctness
- clarity: understandable and relevant
- structure: organization and requested format

Use 1 for very poor and 10 for excellent.
Return JSON only in this format:
{{"instruction_adherence": 1, "accuracy": 1, "clarity": 1, "structure": 1}}
"""

    raw, _, _ = generate_text(
        tokenizer,
        model,
        evaluation_prompt,
        max_new_tokens=100
    )

    scores = extract_json(raw)

    keys = [
        "instruction_adherence",
        "accuracy",
        "clarity",
        "structure"
    ]

    if not isinstance(scores, dict):
        return {key: None for key in keys}

    clean_scores = {}

    for key in keys:
        try:
            value = int(scores[key])
            clean_scores[key] = value if 1 <= value <= 10 else None
        except (KeyError, TypeError, ValueError):
            clean_scores[key] = None

    return clean_scores

In [ ]:
RESULTS_DIR = Path.cwd() / "llm-test" / "P1-A7-results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_FILE = RESULTS_DIR / "model_benchmark_results.csv"
QUALITY_FILE = RESULTS_DIR / "prompt_quality_scores.csv"

In [ ]:
benchmark_rows = []

if BENCHMARK_FILE.exists():
    existing_df = pd.read_csv(BENCHMARK_FILE)
    successful_df = existing_df[existing_df["status"] == "ok"]
    completed = set(
        zip(successful_df["model_id"], successful_df["prompt_id"])
    )
else:
    existing_df = pd.DataFrame()
    completed = set()

for model_info in MODEL_CATALOG:
    model_id = model_info["id"]

    pending_prompts = [
        prompt for prompt in PROMPTS
        if (model_id, prompt["id"]) not in completed
    ]

    if not pending_prompts:
        print("Already completed:", model_info["name"])
        continue

    print(f"\nTesting {model_info['name']} ({model_id})")

    tokenizer = model = None

    try:
        tokenizer, model = load_model(model_id)

        for prompt_item in pending_prompts:
            try:
                raw, seconds, generated_tokens = generate_text(
                    tokenizer,
                    model,
                    prompt_item["prompt"],
                    max_new_tokens=256
                )

                scores = evaluate_response(
                    tokenizer,
                    model,
                    prompt_item["prompt"],
                    raw
                )

                valid_scores = [
                    scores[key] for key in scores
                    if scores[key] is not None
                ]

                quality_score = (
                    sum(valid_scores) / len(valid_scores)
                    if len(valid_scores) == 4
                    else None
                )

                row = {
                    "model": model_info["name"],
                    "model_id": model_id,
                    "category": model_info["category"],
                    "params_b": model_info["params_b"],
                    "prompt_id": prompt_item["id"],
                    "prompt_type": prompt_item["type"],
                    "prompt": prompt_item["prompt"],
                    "raw_output": raw,
                    "latency_seconds": round(seconds, 3),
                    "generated_tokens": generated_tokens,
                    "tokens_per_second": (
                        round(generated_tokens / seconds, 3)
                        if seconds > 0 else None
                    ),
                    **scores,
                    "quality_score": quality_score,
                    "status": "ok",
                    "error": None
                }

            except Exception as exc:
                row = {
                    "model": model_info["name"],
                    "model_id": model_id,
                    "category": model_info["category"],
                    "params_b": model_info["params_b"],
                    "prompt_id": prompt_item["id"],
                    "prompt_type": prompt_item["type"],
                    "prompt": prompt_item["prompt"],
                    "raw_output": None,
                    "latency_seconds": None,
                    "generated_tokens": None,
                    "tokens_per_second": None,
                    "instruction_adherence": None,
                    "accuracy": None,
                    "clarity": None,
                    "structure": None,
                    "quality_score": None,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}"
                }

            # Save immediately after each prompt.
            pd.DataFrame([row]).to_csv(
                BENCHMARK_FILE,
                mode="a",
                header=not BENCHMARK_FILE.exists(),
                index=False
            )

            if row["status"] == "ok":
                completed.add((model_id, prompt_item["id"]))

            print(
                prompt_item["id"],
                "status:", row["status"],
                "latency:", row["latency_seconds"]
            )


    except Exception as exc:
        print(
            f"ERROR on {prompt_item['id']}: "
            f"{type(exc).__name__}: {exc}"
        )
        raise


    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if torch.backends.mps.is_available():
            try:
                torch.mps.empty_cache()
            except Exception:
                pass

print("\nSaved benchmark results to:", BENCHMARK_FILE)

## Part 1: Benchmark Analysis

This section analyzes the benchmark results saved from running the 15 models on the 25 prompts. It compares response quality, latency, and generation speed. Some models may have missing quality scores if the evaluator did not return valid scores.

In [ ]:
# Load the benchmark results
df = pd.read_csv(BENCHMARK_FILE)

print("Total rows:", len(df))
print("Unique models:", df["model"].nunique())
print("Unique prompts:", df["prompt_id"].nunique())

print("\nRows by model:")
display(df.groupby("model").size().reset_index(name="rows"))

print("\nSuccessful and failed runs:")
display(df["status"].value_counts(dropna=False))

print("\nMissing quality scores by model:")
display(
    df.groupby("model")["quality_score"]
      .apply(lambda scores: scores.isna().sum())
      .reset_index(name="missing_quality_scores")
)

In [ ]:
# Keep successful generations for performance analysis
successful = df[df["status"] == "ok"].copy()

model_summary = (
    successful.groupby(
        ["model", "category", "params_b"], as_index=False
    )
    .agg(
        prompts_run=("prompt_id", "nunique"),
        avg_quality=("quality_score", "mean"),
        avg_latency=("latency_seconds", "mean"),
        avg_tokens_per_sec=("tokens_per_second", "mean"),
        scored_responses=("quality_score", "count")
    )
)

model_summary = model_summary.sort_values(
    "avg_quality", ascending=False, na_position="last"
).round(2)

display(model_summary)

# Save the summary for use in your report
model_summary.to_csv(
    RESULTS_DIR / "model_summary.csv", index=False
)

In [ ]:
category_summary = (
    successful.groupby("prompt_type", as_index=False)
    .agg(
        avg_quality=("quality_score", "mean"),
        avg_latency=("latency_seconds", "mean"),
        avg_tokens_per_sec=("tokens_per_second", "mean"),
        scored_responses=("quality_score", "count")
    )
    .round(2)
    .sort_values("avg_quality", ascending=False)
)

display(category_summary)

category_summary.to_csv(
    RESULTS_DIR / "category_summary.csv", index=False
)

In [ ]:
model_by_category = successful.pivot_table(
    index="model",
    columns="prompt_type",
    values="quality_score",
    aggfunc="mean"
).round(2)

display(model_by_category)

In [ ]:
size_summary = (
    successful.groupby("category", as_index=False)
    .agg(
        avg_quality=("quality_score", "mean"),
        avg_latency=("latency_seconds", "mean"),
        avg_tokens_per_sec=("tokens_per_second", "mean"),
        scored_responses=("quality_score", "count"),
        models_tested=("model", "nunique")
    )
    .round(2)
)

display(size_summary)

size_summary.to_csv(
    RESULTS_DIR / "size_category_summary.csv", index=False
)

In [ ]:
import matplotlib.pyplot as plt

plot_data = model_summary.dropna(
    subset=["params_b", "avg_quality"]
)

plt.figure(figsize=(9, 5))
plt.scatter(plot_data["params_b"], plot_data["avg_quality"])

for _, row in plot_data.iterrows():
    plt.annotate(
        row["model"],
        (row["params_b"], row["avg_quality"]),
        fontsize=7,
        xytext=(4, 4),
        textcoords="offset points"
    )

plt.xlabel("Model Size (Billions of Parameters)")
plt.ylabel("Average Quality Score")
plt.title("Model Size vs. Average Quality")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "model_size_vs_quality.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
plot_data = model_summary.dropna(
    subset=["avg_quality", "avg_tokens_per_sec"]
)

plt.figure(figsize=(9, 5))
plt.scatter(
    plot_data["avg_tokens_per_sec"],
    plot_data["avg_quality"]
)

for _, row in plot_data.iterrows():
    plt.annotate(
        row["model"],
        (row["avg_tokens_per_sec"], row["avg_quality"]),
        fontsize=7,
        xytext=(4, 4),
        textcoords="offset points"
    )

plt.xlabel("Average Generation Speed (Tokens/Second)")
plt.ylabel("Average Quality Score")
plt.title("Generation Speed vs. Quality")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "speed_vs_quality.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
score_columns = [
    "instruction_adherence",
    "accuracy",
    "clarity",
    "structure"
]

score_completeness = (
    df.groupby("model")[score_columns]
    .agg(
        lambda column: column.notna().sum()
    )
)

score_completeness["complete_score_rows"] = (
    df.groupby("model")["quality_score"]
    .apply(lambda column: column.notna().sum())
)

display(score_completeness)

score_completeness.to_csv(
    RESULTS_DIR / "score_completeness.csv"
)

In [ ]:
valid_models = model_summary.dropna(subset=["avg_quality"])

if not valid_models.empty:
    best_quality = valid_models.loc[
        valid_models["avg_quality"].idxmax()
    ]

    print(
        f"Highest average quality: {best_quality['model']} "
        f"({best_quality['avg_quality']:.2f})"
    )

valid_speed = model_summary.dropna(
    subset=["avg_tokens_per_sec"]
)

if not valid_speed.empty:
    fastest = valid_speed.loc[
        valid_speed["avg_tokens_per_sec"].idxmax()
    ]

    print(
        f"Highest average throughput: {fastest['model']} "
        f"({fastest['avg_tokens_per_sec']:.2f} tokens/sec)"
    )

valid_latency = model_summary.dropna(subset=["avg_latency"])

if not valid_latency.empty:
    lowest_latency = valid_latency.loc[
        valid_latency["avg_latency"].idxmin()
    ]

    print(
        f"Lowest average generation latency: "
        f"{lowest_latency['model']} "
        f"({lowest_latency['avg_latency']:.2f} seconds)"
    )

print("\nModels with incomplete quality scores:")
display(
    model_summary[
        model_summary["scored_responses"]
        < model_summary["prompts_run"]
    ][["model", "prompts_run", "scored_responses"]]
)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lrangu/PycharmProjects/WelcomeScreen/14_ClassCompass/llm-test/P1-A7-results/model_summary.csv'